## Full Fact Data Load

In this notebook we will perform full fact data load of the historical data of the child company into parent data, here we will load, clean, standardise the data before upserting the data to the exsitng fact table

### Setup Environment 

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

## Define Environment Variables and Path

In [0]:
%run /Workspace/Users/kavyajeetbora@gmail.com/fmcg_databricks/consolidated_pipeline/dimension_data_processing/00_config

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")
dbutils.widgets.text("s3", "sportsbar-raw-data-5684", "S3 Bucket")

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')
s3_bucket = dbutils.widgets.get('s3')

## s3 Basepath
base_path = f"s3://{s3_bucket}/{data_source}"
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"

## define the tables in databricks
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

### Batch Load Data from Landing

In [0]:
df = (
    spark.read.format("csv")
    .options(header=True, inferSchema=True)
    .load(f"{landing_path}/*.csv")
)

df = (
    df.withColumn("read_timestamp", F.current_timestamp())
    .select(
        "*",
        "_metadata.file_name",
        "_metadata.file_size"
    )
)

print(f"Total Rows: {df.count()}")
display(df.limit(4))

### Load the Data to Bronze Table

In [0]:
(
    df.write
    .format("delta")
    .option("delta.enableChangeDataFeed", True)
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

print(f"Loaded the data to the table: {bronze_table}")
df_bronze_count = spark.sql(f"SELECT COUNT(*) FROM {bronze_table}").first()[0]
print(f"Total Records added : {df_bronze_count}")

### Moving files from source to processed directory

In [0]:
from tqdm.notebook import tqdm

In [0]:
files = dbutils.fs.ls(landing_path)
for file_info in tqdm(files):
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

### Silver

Here we will transform the data: 
1. Cleaning
2. Handle null values
3. Standardizing and formatting data

In [0]:
df_silver = spark.sql(f"SELECT * FROM {bronze_table}")
df_silver.show(2)

- Remove any rows where the order quantity is missing or null.
- Change any non-numeric customer IDs to "999999" and format the column as a string.
- Remove the weekday name from the beginning of the order placement date text.
- Convert the order placement date into a standard date format by parsing it against multiple possible layouts.
- Remove duplicate rows based on order ID, date, customer ID, product ID, and order quantity.
- Convert the product ID column data type to a string.

In [0]:
## Cleaning Process

# 1. Remove any rows where the order quantity is missing or null.
df_silver = df_silver.filter(F.col("order_qty").isNotNull())

# 2. Change any non-numeric customer IDs to "999999" and format the column as a string.
df_silver = (
    df_silver
    .withColumn(
        "customer_id",
        F.when(
            F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id")
        )
        .otherwise(
            F.lit("999999").cast("string")
        )
    )
)

## Remove the weekday name from the beginning of the order placement date text.
df_silver = df_silver.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# Convert the order placement date into a standard date format by parsing it against multiple possible layouts.
df_silver = (
    df_silver
    .withColumn(
        "order_placement_date",
        F.coalesce(
            F.try_to_date("order_placement_date", "yyyy/MM/dd"),
            F.try_to_date("order_placement_date", "dd-MM-yyyy"),
            F.try_to_date("order_placement_date", "dd/MM/yyyy"),
            F.try_to_date("order_placement_date", "MMMM dd, yyyy")
        )
    )
)

# Remove duplicate rows based on order ID, date, customer ID, product ID, and order quantity.
df_silver = df_silver.drop_duplicates(subset=['order_id', "customer_id", 'product_id', 'order_qty'])

# Convert the product ID column data type to a string.
df_silver = (
    df_silver.withColumn("product_id", F.col("product_id").cast("string"))
)

In [0]:
display(df_silver.limit(10))

In [0]:
# check what's the maximum and minimum date
df_silver.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

## Assign product code

In [0]:
df_products = spark.table("fmcg.silver.products")

In [0]:
df_products = spark.table("fmcg.silver.products")

df_silver_v2 = (
    df_silver
    .join(
        other = df_products.select("product_id", "product_code"),
        on = "product_id",
        how = "inner"
    )
)
display(df_silver_v2.limit(10))

## Write to Catalog: `silver_table`

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    (
        df_silver_v2.write.format('delta')
        .option('enableChangeDataFeed', True)
        .option("mergeSchema", True)
        .mode('overwrite')
        .saveAsTable(silver_table)
    )
else: 
    silver_delta = DeltaTable.forName(spark, silver_table)
    (
        silver_delta.alias('silver')
        .merge(
            df_silver_v2.alias('bronze'),
            """
            silver.order_placement_date = bronze.order_placement_date 
            AND silver.order_id = bronze.order_id
            AND silver.product_id = bronze.product_id
            AND silver.customer_id = bronze.customer_id
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

## GOLD

- Write the child dataset to gold schema

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")
print("TOTAL COUNT:",df_gold.count())
display(df_gold.limit(4))

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print(f"creating new table: {gold_table}...")
    
    (
        df_gold.write.format('delta')
        .option('mergeSchema', True)
        .option('enableChangeDataFeed', True)
        .mode('overwrite')
        .saveAsTable(gold_table)
    )

else:
    ## if there is anychange or update in the silver table, then update the gold table
    gold_delta = DeltaTable.forName(spark, gold_table)

    ## Then upsert the records
    (
        gold_delta.alias("gold")
        .merge(
            df_gold.alias("silver"),
            """
            gold.date = silver.date
            AND gold.order_id = silver.order_id
            AND gold.product_code = silver.product_code
            AND gold.customer_code = silver.customer_code
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

## Merging the Child Gold Table to Parent Gold Table

- The Child table is present at Daily Grain whereas Parent is at Monthly Grain
- Aggregate the Child Table to monthly Grain and Then merge it with Parent

In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_child.show(3)

In [0]:
display(df_child.select('date').distinct().orderBy(F.col('date').asc()).limit(10))

Since the child table is in daily grain -> aggregate to monthly grain

In [0]:
df_orders_master = spark.sql("SELECT * FROM fmcg.gold.fact_orders")
df_orders_master.columns

In [0]:
df_monthly = (
    df_child
    .withColumn(
        "MTD",
        F.date_trunc("MM", F.col("date"))
    )
    .groupBy("MTD", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )
    .withColumnRenamed("MTD", "date")
)

In [0]:
display(df_monthly.limit(5))

In [0]:
## Merge the Parent with Child
parent_gold_table = f"{catalog}.{gold_schema}.fact_orders"
gold_parent = DeltaTable.forName(spark, parent_gold_table)
(
    gold_parent.alias("gold")
    .merge(
        df_monthly.alias("gold_child"),
        """
            gold.customer_code == gold_child.customer_code
            AND gold.product_code == gold_child.product_code
            AND gold.date == gold_child.date
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
fact_oders = spark.sql(f"SELECT * from {parent_gold_table}")
display(fact_oders.limit(5))